# Dutch publication Docling preparation

This notebook validates the Stage 230 data-preparation handoff for the Wet open overheid smoke document. It checks each processing step before the same contract is automated with the DSPA/KFP Docling pipeline and used by the RAG smoke test.

## 1. Confirm curated workspace content

The visible workspace contains notebooks only. Pipeline source, scripts, and sample data are under hidden `.stage230` content generated during workbench startup.

In [ ]:
from pathlib import Path

workspace = Path.cwd()
stage = workspace / '.stage230'
assert (stage / 'kfp/dutch_publication_docling_pipeline.py').exists(), 'missing KFP pipeline source'
assert (stage / 'scripts/dutch_publication_prepare.py').exists(), 'missing preparation helper'
assert (stage / 'data/dutch-government/source/stb-2022-14.pdf').exists(), 'missing source PDF'
assert (stage / 'data/dutch-government/metadata/stb-2022-14-metadata.json').exists(), 'missing metadata contract'
print('Workspace ready:', workspace)
print('Generated helper area:', stage)

## 2. Compile the Docling KFP pipeline

The pipeline adapts the Red Hat `opendatahub-io/data-processing` `docling-standard` pattern for this single PDF. Compiled YAML is generated as a local artifact; the source pipeline is committed and the repository runner imports versions into the GitOps-managed DSPA server.

In [ ]:
import subprocess
import sys

compiled = stage / 'compiled/stage-230-dutch-publication-docling.yaml'
compiled.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([
    sys.executable,
    str(stage / 'kfp/dutch_publication_docling_pipeline.py'),
    '--output', str(compiled),
], check=True)
print('Compiled pipeline:', compiled)
print('Size:', compiled.stat().st_size, 'bytes')

## 3. Validate the chunk contract

The KFP task uses Docling. This local notebook cell uses `pypdf` by default only to validate article detection and metadata shaping inside the regular workbench. The supported automation gate runs the same contract with Docling through DSPA/KFP.

In [ ]:
import os

converter = os.environ.get('RHOAI_STAGE230_PREP_CONVERTER', 'pypdf')
prepared = stage / 'data/dutch-government/processed/stb-2022-14-docling-chunks.jsonl'
subprocess.run([
    sys.executable,
    str(stage / 'scripts/dutch_publication_prepare.py'),
    '--converter', converter,
    '--source-pdf', str(stage / 'data/dutch-government/source/stb-2022-14.pdf'),
    '--metadata', str(stage / 'data/dutch-government/metadata/stb-2022-14-metadata.json'),
    '--output', str(prepared),
    '--converted-dir', str(stage / 'data/dutch-government/processed/docling'),
], check=True)
print('Prepared chunks:', prepared)

## 4. Run the RAG smoke test against prepared chunks

This uses the same metadata extraction, filtered hybrid search, reranking, and Nemotron answer path as the normal Dutch smoke test.

In [ ]:
subprocess.run([
    sys.executable,
    str(stage / 'scripts/dutch_publication_rag_smoke.py'),
    '--reset',
    '--sample', str(prepared),
    '--vector-store', 'stage230-dutch-woo-docling-demo',
    '--search-mode', 'hybrid',
    '--query', 'Binnen welke termijn moet een bestuursorgaan beslissen op een verzoek om informatie?',
    '--expected-topic', 'openbaarmaking_op_verzoek',
    '--expected-term', 'vier weken',
], check=True)